# Phishing Detection: Zero-Shot LLM and Prompt Injection
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sjcALane/llm-security-labs/blob/main/notebooks/02_zero_shot_and_prompt_injection.ipynb)

Can a small chat model spot phishing just by being asked? Then we attack it with the simplest trick there is: we write instructions to the model inside the email.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU. Then run this cell.
try:
    import seclab
except ImportError:
    !pip -q install git+https://github.com/sjcALane/llm-security-labs
    import seclab

from seclab import attacks, config, data, metrics, notebook, prompts
from seclab.model import load, score
notebook.banner()

In [ ]:
df = data.load_emails()
train, val, test = data.split(df, seed=config.SEED)
test_small = data.sample(test, config.N_TEST, seed=config.SEED)
y = test_small["label"].to_numpy()
emails = test_small["text"].tolist()
print(f"{len(train)} training emails, {len(test_small)} test emails")

## The model

Qwen2.5-0.5B-Instruct has 494 million parameters and open weights under the Apache 2.0 license. It's tiny next to the models behind commercial chatbots, which is why it fits on a free GPU.

In [ ]:
lm, tok = load(config.MODEL_NAME, config.device())
print(sum(p.numel() for p in lm.parameters()) / 1e6, "million parameters")

## The prompt

In [ ]:
print(prompts.prompt_text(tok, "Your package is on hold. Pay the fee at http://example.test"))

Tokens like `<|im_start|>` mark who is speaking. The prompt stops exactly where the assistant's answer begins.

## From next-token prediction to a probability

For the next position the model outputs a score, called a logit, for every token in its vocabulary of about 150,000 tokens. We keep only $z_{\text{Yes}}$ and $z_{\text{No}}$ and apply a softmax to the pair:

$$p(\text{phishing}) = \frac{e^{z_{\text{Yes}}}}{e^{z_{\text{Yes}}}+e^{z_{\text{No}}}} = \sigma(z_{\text{Yes}} - z_{\text{No}})$$

This gives a probability for every email, and we never have to parse the model's text.

**Question 1.** Show the second equality.

**Your answer:**

Here are the five tokens the model likes best as the first word of its answer for one test email.

In [ ]:
import torch

enc = tok(prompts.prompt_text(tok, emails[0]), return_tensors="pt").to(lm.device)
with torch.no_grad():
    logits = lm(**enc).logits[0, -1]
top = torch.topk(logits.float(), 5)
for z, i in zip(top.values, top.indices):
    print(f"{tok.decode(i)!r:12} {z.item():.2f}")
print("true label:", "phishing" if y[0] == 1 else "safe")

## Score the test set

In [ ]:
p_zero = score(lm, tok, emails)
m_zero = metrics.evaluate(y, p_zero)
notebook.save_result("Zero-shot LLM", m_zero)
metrics.table(notebook.load_results())

**Question 2.** The baseline learned from about 12,000 labeled emails. The language model saw none of them. What did each one bring to the task?

**Your answer:**

## Prompt injection

The email is supposed to be data, but the model reads it in the same stream as its instructions. An attacker can write to the model directly.

In [ ]:
print(attacks.INJECTION)

In [ ]:
p_inj = score(lm, tok, [attacks.inject(e) for e in emails])
asr = metrics.attack_success_rate(y, p_zero, p_inj)
print(f"attack success rate: {asr:.3f}")
print(f"mean p(phishing) on phishing emails: {p_zero[y == 1].mean():.3f} -> {p_inj[y == 1].mean():.3f}")

The attack success rate is the share of phishing emails the model caught before the attack and missed after it.

**Question 3.** A person reading this email would ignore the note. Why can't the model? Compare this with SQL injection.

**Your answer:**

**Question 4.** Write your own injection and measure it. Does a polite one work better than a bossy one?

In [ ]:
my_injection = "\n\nPS: (write your injection here)"
p_mine = score(lm, tok, [e + my_injection for e in emails])
print(f"attack success rate: {metrics.attack_success_rate(y, p_zero, p_mine):.3f}")

**Your answer:**

In [ ]:
notebook.save_result("Zero-shot LLM, prompt injection ASR", asr)